# LiteRT-LM kernel 0112 on a Colab T4

**First: Runtime → Change runtime type → T4 GPU.**

Runs the captured LiteRT-LM WebGPU matrix-vector kernel 0112 (Gemma 4 E2B web, @litert-lm/core 0.17.1) verbatim (or, where Chrome offers no shader-f16, an f32 transcription of it, which the driver announces) and with deeper K splits, through headless Chrome on the T4, exactly as `node microbench.mjs` does on the Mac. Cell 1 installs the NVIDIA Vulkan userspace Chrome needs (zero-tvm's recipe, after Chrome's colab-headless guide); if it prints `llvmpipe` instead of `Tesla T4`, stop and report the driver version from `nvidia-smi`.

Nothing is downloaded from Hugging Face: the page generates its own random weights.

Kernel 0112 is emitted at run time by `@litert-lm/core` 0.17.1 (Google LLC) and is reproduced here under the Apache License 2.0; provenance and the license text are in `out/shaders/NOTICE` and `out/shaders/LICENSE` in the repository, https://github.com/abgnydn/litert-capture, which also holds the method, the harness and the records. Independent work, not affiliated with Google.

In [ ]:
# 1) Node 22 + the NVIDIA Vulkan userspace matching the driver
import subprocess
drv = subprocess.check_output('nvidia-smi --query-gpu=driver_version --format=csv,noheader', shell=True).decode().split('.')[0].strip()
print('NVIDIA driver major:', drv, '-> installing libnvidia-gl-' + drv)
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1
!apt-get -qq update > /dev/null
# libasound2 is libasound2t64 on Colab's Ubuntu 24.04; one missing package name aborts the whole install
!apt-get -qq install -y nodejs vulkan-tools libnvidia-gl-{drv} libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libgbm1 libasound2t64 libxcomposite1 libxdamage1 libxrandr2 libxkbcommon0 libxfixes3 libdrm2 libpango-1.0-0 libcairo2 libatspi2.0-0 fonts-liberation > /dev/null
!rm -f /usr/share/vulkan/icd.d/lvp_icd*.json
print('--- Vulkan device (want Tesla T4, not llvmpipe): ---')
!vulkaninfo --summary 2>/dev/null | grep -E 'deviceName|driverName' || echo 'NO VULKAN DEVICE'

In [ ]:
# 2) Unpack the page, the captured kernel and the driver; install puppeteer 25.11.0 (downloads Chrome for Testing 153, which made the committed T4 record)
import base64, os, json
files = {"microbench.html":"PCFkb2N0eXBlIGh0bWw+CjxtZXRhIGNoYXJzZXQ9InV0Zi04Ij4KPHRpdGxlPkxpdGVSVC1MTSBrZXJuZWwgMDExMiBtaWNyb2JlbmNobWFyazwvdGl0bGU+CjxwcmUgaWQ9ImxvZyIgc3R5bGU9IndoaXRlLXNwYWNlOiBwcmUtd3JhcDsgZm9udDogMTJweCBtb25vc3BhY2UiPjwvcHJlPgoKPCEtLSBSdW5zIHRoZSBjYXB0dXJlZCAyLWJpdCBtYXRyaXgtdmVjdG9yIGtlcm5lbCAwMTEyIGluIGlzb2xhdGlvbiBvbiBpdHMgcmVhbAogICAgIGRlY29kZSBzaGFwZSAoMTUzNiAtPiAxMjI4OCwgNC41IE1CIG9mIHBhY2tlZCB3ZWlnaHRzKSBhbmQgY29tcGFyZXMgaXQKICAgICBhZ2FpbnN0IG1pbmltYWwgdGV4dCBlZGl0cyBvZiB0aGUgc2FtZSBzb3VyY2UuIFR3byB2YXJpYWJsZXMgYXJlIGtlcHQKICAgICBzZXBhcmF0ZSBvbiBwdXJwb3NlLCBiZWNhdXNlIGEgdmFyaWFudCB0aGF0IGNoYW5nZXMgYm90aCBjYW5ub3Qgc2F5IHdoaWNoIG1vdmVkIHRoZSBudW1iZXI6CiAgICAgICBpbnZvY2F0aW9ucyBwZXIgd29ya2dyb3VwICg2NCBhcyBzaGlwcGVkLCBvciAyNTYpCiAgICAgICB0b3RhbCB0aHJlYWRzIGRpc3BhdGNoZWQgICAoMTIsMjg4IGFzIHNoaXBwZWQsIG9yIDR4IC8gOHggdmlhIGEgZGVlcGVyIEsgc3BsaXQpCiAgICAgVmFyaWFudHM6CiAgICAgICBvcmlnICAgICAgICAxNng0ICB3ZyAoNjQgaW52KSwgIDEyLDI4OCB0aHJlYWRzLCB0ZXh0dXJlIHdlaWdodHMgICA8LSBhcyBzaGlwcGVkCiAgICAgICB3ZzI1Ni1zYW1lICA2NHg0ICB3ZyAoMjU2IGludiksIDEyLDI4OCB0aHJlYWRzICAgICAgICAgICAgICAgICAgICA8LSBjb250cm9sOiBiaWdnZXIgd29ya2dyb3VwIG9ubHkKICAgICAgIHNwbGl0MTYtNjQgIDR4MTYgIHdnICg2NCBpbnYpLCAgNDksMTUyIHRocmVhZHMgICAgICAgICAgICAgICAgICAgIDwtIGNvbnRyb2w6IG1vcmUgdGhyZWFkcyBvbmx5CiAgICAgICBzcGxpdDMyLTY0ICAyeDMyICB3ZyAoNjQgaW52KSwgIDk4LDMwNCB0aHJlYWRzICAgICAgICAgICAgICAgICAgICA8LSBzYW1lIHdnIHNoYXBlIGFzIHRoZWlyIGtlcm5lbCAwMTEzCiAgICAgICBzcGxpdDE2LTI1NiAxNngxNiB3ZyAoMjU2IGludiksIDQ5LDE1MiB0aHJlYWRzICAgICAgICAgICAgICAgICAgICA8LSBib3RoCiAgICAgICBidWYtNCAgICAgICAxNng0LCAxMiwyODggdGhyZWFkcywgd2VpZ2h0cyBmcm9tIGEgc3RvcmFnZSBidWZmZXIKICAgICBNZWFzdXJlbWVudCBvcmRlcjogYSBkaXNjYXJkZWQgd2FybS11cCB2YXJpYW50IGZpcnN0ICh0aGUgZmlyc3QgdGltZWQgc2xvdAogICAgIHJlYWRzIHNsb3cpLCB0aGVuIGV2ZXJ5IHZhcmlhbnQgZm9yd2FyZCBhbmQgYWdhaW4gaW4gcmV2ZXJzZSBvcmRlcjsgdGhlCiAgICAgY29sZCBhbmQgaG90IHNhbXBsZXMgZnJvbSBib3RoIHBvc2l0aW9ucyBhcmUgcG9vbGVkIGJlZm9yZSB0aGUgbWVkaWFuLgogICAgICJDb2xkIiByb3RhdGVzIHRocm91Z2ggMjQgbWF0cmljZXMgKDEwOCBNQikgc28gd2VpZ2h0cyBjb21lIGZyb20gRFJBTTsKICAgICAiaG90IiByZXVzZXMgb25lIG1hdHJpeC4gRXZlcnkgdmFyaWFudCBpcyBjaGVja2VkIGFnYWluc3QgYSBDUFUgcmVmZXJlbmNlLiAtLT4KPHNjcmlwdCB0eXBlPSJtb2R1bGUiPgogIGNvbnN0IGxvZyA9IChzKSA9PiB7IGRvY3VtZW50LmdldEVsZW1lbnRCeUlkKCdsb2cnKS5hcHBlbmQocyArICdcbicpOyBjb25zb2xlLmxvZygnW21iXSAnICsgcykgfQogIGNvbnN0IE9VVF9TTElDRVMgPSAzMDcyLCBJTl9TTElDRVMgPSAzODQgICAgICAvLyBVLmkwID0gWzMwNzIsIDM4NF06IDEyMjg4IG91dHB1dHMsIDE1MzYgaW5wdXRzCiAgY29uc3QgV0VJR0hUX0JZVEVTID0gT1VUX1NMSUNFUyAqIElOX1NMSUNFUyAqIDQgLy8gb25lIHJnYmE4dWludCB0ZXhlbCBwZXIgKGRzdF9zLCBzcmNfcykKICBjb25zdCBOTUFUX0NPTEQgPSAyNCwgUk9VTkRTID0gMTIKICAvLyA/ZjMyPTE6IHJ1biBhbiBmMzIgdHJhbnNjcmlwdGlvbiBvZiB0aGUga2VybmVsIChmMTYgLT4gZjMyIHRocm91Z2hvdXQsIHNjYWxlcwogIC8vIHVwbG9hZGVkIGFzIGYzMikgZm9yIGFkYXB0ZXJzIHRoYXQgZG8gbm90IGV4cG9zZSBzaGFkZXItZjE2LiBOb3QgR29vZ2xlJ3MKICAvLyBrZXJuZWwgdmVyYmF0aW07IHNhbWUgc3RydWN0dXJlLCBzbyBpdCBzdGlsbCB0ZXN0cyB0aGUgSy1zcGxpdCBsZXZlci4KICBjb25zdCBGMzIgPSBuZXcgVVJMU2VhcmNoUGFyYW1zKGxvY2F0aW9uLnNlYXJjaCkuZ2V0KCdmMzInKSA9PT0gJzEnCiAgY29uc3QgVjQgPSBGMzIgPyAndmVjNDxmMzI+JyA6ICd2ZWM0PGYxNj4nCiAgY29uc3QgTkVFRCA9IEYzMiA/IFsndGltZXN0YW1wLXF1ZXJ5J10gOiBbJ3NoYWRlci1mMTYnLCAndGltZXN0YW1wLXF1ZXJ5J10KICBjb25zdCBSID0gKHdpbmRvdy5fX21iID0geyB2YXJpYW50czoge30sIHdlaWdodEJ5dGVzOiBXRUlHSFRfQllURVMsIG9yZGVyOiBbXSwgZjMyOiBGMzIgfSkKCiAgY29uc3QgZjE2ID0gKHgpID0+IHsgY29uc3QgZiA9IG5ldyBGbG9hdDMyQXJyYXkoW3hdKSwgdSA9IG5ldyBVaW50MzJBcnJheShmLmJ1ZmZlcilbMF07IGNvbnN0IHMgPSAodSA+PiAxNikgJiAweDgwMDAsIGUgPSAoKHUgPj4gMjMpICYgMHhmZikgLSAxMTIsIG0gPSB1ICYgMHg3ZmZmZmY7IGlmIChlIDw9IDApIHJldHVybiBzOyBpZiAoZSA+PSAzMSkgcmV0dXJuIHMgfCAweDdjMDA7IGxldCBoID0gcyB8IChlIDw8IDEwKSB8IChtID4+IDEzKTsgaWYgKChtICYgMHgxZmZmKSA+IDB4MTAwMCB8fCAoKG0gJiAweDFmZmYpID09PSAweDEwMDAgJiYgKGggJiAxKSkpIGgrKzsgcmV0dXJuIGggfQogIGNvbnN0IGZyb21GMTYgPSAoaCkgPT4geyBjb25zdCBzID0gaCAmIDB4ODAwMCA/IC0xIDogMSwgZSA9IChoID4+IDEwKSAmIDB4MWYsIG0gPSBoICYgMHgzZmY7IHJldHVybiBlID09PSAwID8gcyAqIG0gKiAyICoqIC0yNCA6IGUgPT09IDMxID8gKG0gPyBOYU4gOiBzICogSW5maW5pdHkpIDogcyAqICgxICsgbSAvIDEwMjQpICogMiAqKiAoZSAtIDE1KSB9CiAgbGV0IHNlZWQgPSAxMjM0NQogIGNvbnN0IHJuZCA9ICgpID0+ICgoc2VlZCA9IChzZWVkICogMTY2NDUyNSArIDEwMTM5MDQyMjMpID4+PiAwKSAvIDIgKiogMzIpCiAgY29uc3QgbWVkaWFuID0gKGEpID0+IHsgY29uc3QgcyA9IFsuLi5hXS5zb3J0KCh4LCB5KSA9PiB4IC0geSk7IHJldHVybiBzW01hdGguZmxvb3Iocy5sZW5ndGggLyAyKV0gfQogIGNvbnN0IHBjdCA9IChhLCBwKSA9PiB7IGNvbnN0IHMgPSBbLi4uYV0uc29ydCgoeCwgeSkgPT4geCAtIHkpOyByZXR1cm4gc1tNYXRoLm1pbihzLmxlbmd0aCAtIDEsIE1hdGguZmxvb3Iocy5sZW5ndGggKiBwKSldIH0KCiAgY29uc3QgbXVzdFJlcGxhY2UgPSAoc3JjLCBmcm9tLCB0bykgPT4geyBpZiAoIXNyYy5pbmNsdWRlcyhmcm9tKSkgdGhyb3cgbmV3IEVycm9yKCdrZXJuZWwgdGV4dCBjaGFuZ2VkLCBhbmNob3Igbm90IGZvdW5kOiAnICsgZnJvbS5zbGljZSgwLCA2MCkpOyByZXR1cm4gc3JjLnNwbGl0KGZyb20pLmpvaW4odG8pIH0KICBjb25zdCB2YXJpYW50U291cmNlID0gKGJhc2UsIGJ1Ziwga3MsIHdnWCkgPT4gewogICAgbGV0IHMgPSBiYXNlCiAgICBpZiAoYnVmKSB7CiAgICAgIHMgPSBtdXN0UmVwbGFjZShzLCAnQGdyb3VwKDApIEBiaW5kaW5nKDIpIHZhciB3ZWlnaHRzX2ltYWdlMmQgOiB0ZXh0dXJlXzJkPHUzMj47JywKICAgICAgICAnc3RydWN0IHdlaWdodHNfdTMyX3ZlY3RvciB7IGRhdGE6IGFycmF5PHUzMj4sIH07XG5AZ3JvdXAoMCkgQGJpbmRpbmcoMikgdmFyPHN0b3JhZ2UsIHJlYWQ+IHdlaWdodHNfdTMyIDogd2VpZ2h0c191MzJfdmVjdG9yOycpCiAgICAgIHMgPSBtdXN0UmVwbGFjZShzLCAndmFyIHcgOiB2ZWM0PHUzMj49IHZlYzQ8dTMyPih0ZXh0dXJlTG9hZCh3ZWlnaHRzX2ltYWdlMmQsIHZlYzI8aTMyPigoZHN0X3MpLCAoc3JjX3MpKSwgMCkpOycsCiAgICAgICAgJ3ZhciB3cCA6IHUzMiA9IHdlaWdodHNfdTMyLmRhdGFbc3JjX3MgKiBVLmkwLnggKyBkc3Rfc107XG4gICAgdmFyIHcgOiB2ZWM0PHUzMj49IHZlYzQ8dTMyPih3cCAmIDB4RkZ1LCAod3AgPj4gOHUpICYgMHhGRnUsICh3cCA+PiAxNnUpICYgMHhGRnUsICh3cCA+PiAyNHUpICYgMHhGRnUpOycpCiAgICB9CiAgICBpZiAoa3MgIT09IDQgfHwgd2dYICE9PSAxNikgewogICAgICBzID0gbXVzdFJlcGxhY2UocywgYHZhcjx3b3JrZ3JvdXA+IHRlbXAgOiBhcnJheTwke1Y0fSwgNjQ+O2AsIGB2YXI8d29ya2dyb3VwPiB0ZW1wIDogYXJyYXk8JHtWNH0sICR7d2dYICoga3N9PjtgKQogICAgICBzID0gbXVzdFJlcGxhY2UocywgJ2ZvciAodmFyIHNyY19zIDogaTMyPSB0aWQueTsgc3JjX3MgPCBVLmkwLnk7IHNyY19zICs9IDQpIHsnLCBgZm9yICh2YXIgc3JjX3MgOiBpMzI9IHRpZC55OyBzcmNfcyA8IFUuaTAueTsgc3JjX3MgKz0gJHtrc30pIHtgKQogICAgICBzID0gbXVzdFJlcGxhY2UocywgJ3RlbXBbdGlkLnggKiA0ICsgdGlkLnldJywgYHRlbXBbdGlkLnggKiAke2tzfSArIHRpZC55XWApCiAgICAgIHMgPSBtdXN0UmVwbGFjZShzLCAnZm9yICh2YXIgeXN0cmlkZSA6IGkzMj0gNCAvIDI7IHlzdHJpZGUgPiAwOyB5c3RyaWRlIC89IDIpIHsnLCBgZm9yICh2YXIgeXN0cmlkZSA6IGkzMj0gJHtrc30gLyAyOyB5c3RyaWRlID4gMDsgeXN0cmlkZSAvPSAyKSB7YCkKICAgICAgcyA9IG11c3RSZXBsYWNlKHMsICdyX3NwMCArPSB0ZW1wW3RpZC54ICogNCArIHRpZC55ICsgeXN0cmlkZV07JywgYHJfc3AwICs9IHRlbXBbdGlkLnggKiAke2tzfSArIHRpZC55ICsgeXN0cmlkZV07YCkKICAgIH0KICAgIHJldHVybiBzCiAgfQoKICB0cnkgewogICAgY29uc3QgYWRhcHRlciA9IChhd2FpdCBuYXZpZ2F0b3IuZ3B1LnJlcXVlc3RBZGFwdGVyKHsgcG93ZXJQcmVmZXJlbmNlOiAnaGlnaC1wZXJmb3JtYW5jZScgfSkgPz8gYXdhaXQgbmF2aWdhdG9yLmdwdS5yZXF1ZXN0QWRhcHRlcigpKQogICAgZm9yIChjb25zdCBuZWVkIG9mIE5FRUQpIGlmICghYWRhcHRlci5mZWF0dXJlcy5oYXMobmVlZCkpIHRocm93IG5ldyBFcnJvcignYWRhcHRlciBsYWNrcyAnICsgbmVlZCkKICAgIGNvbnN0IGRldmljZSA9IGF3YWl0IGFkYXB0ZXIucmVxdWVzdERldmljZSh7IHJlcXVpcmVkRmVhdHVyZXM6IE5FRUQgfSkKICAgIGRldmljZS5hZGRFdmVudExpc3RlbmVyKCd1bmNhcHR1cmVkZXJyb3InLCAoZSkgPT4geyBSLmdwdUVycm9yID0gU3RyaW5nKGUuZXJyb3I/Lm1lc3NhZ2UgPz8gZSk7IGxvZygnR1BVIEVSUk9SOiAnICsgUi5ncHVFcnJvcikgfSkKICAgIFIuYWRhcHRlciA9IHsgdmVuZG9yOiBhZGFwdGVyLmluZm8/LnZlbmRvciwgYXJjaGl0ZWN0dXJlOiBhZGFwdGVyLmluZm8/LmFyY2hpdGVjdHVyZSB9CgogICAgbGV0IGJhc2UgPSBhd2FpdCAoYXdhaXQgZmV0Y2goJy9vdXQvc2hhZGVycy8wMTEyX3VubGFiZWxlZC53Z3NsJykpLnRleHQoKQogICAgUi5iYXNlQnl0ZXMgPSBiYXNlLmxlbmd0aAogICAgaWYgKEYzMikgYmFzZSA9IGJhc2UucmVwbGFjZSgnZW5hYmxlIGYxNjsnLCAnJykuc3BsaXQoJ3ZlYzQ8ZjE2PicpLmpvaW4oJ3ZlYzQ8ZjMyPicpLnNwbGl0KCdmMTYoJykuam9pbignZjMyKCcpCgogICAgLy8gZGF0YTogcmFuZG9tIDItYml0IHdlaWdodHMgKHVucGF0dGVybmVkIGJ5dGVzKSwgcGVyLW91dHB1dC1jaGFubmVsIHNjYWxlcywgaW5wdXQgdmVjdG9yCiAgICBjb25zdCB3ZWlnaHRzID0gW10KICAgIGZvciAobGV0IG0gPSAwOyBtIDwgTk1BVF9DT0xEOyBtKyspIHsgY29uc3QgYSA9IG5ldyBVaW50OEFycmF5KFdFSUdIVF9CWVRFUyk7IGZvciAobGV0IGkgPSAwOyBpIDwgYS5sZW5ndGg7IGkrKykgYVtpXSA9IE1hdGguZmxvb3Iocm5kKCkgKiAyNTYpOyB3ZWlnaHRzLnB1c2goYSkgfQogICAgY29uc3Qgc2NhbGUgPSBuZXcgRmxvYXQzMkFycmF5KE9VVF9TTElDRVMgKiA0KTsgZm9yIChsZXQgaSA9IDA7IGkgPCBzY2FsZS5sZW5ndGg7IGkrKykgc2NhbGVbaV0gPSAwLjAxICsgMC4wMSAqIHJuZCgpCiAgICBjb25zdCBpbnB1dCA9IG5ldyBGbG9hdDMyQXJyYXkoSU5fU0xJQ0VTICogNCk7IGZvciAobGV0IGkgPSAwOyBpIDwgaW5wdXQubGVuZ3RoOyBpKyspIGlucHV0W2ldID0gcm5kKCkgKiAyIC0gMQogICAgY29uc3Qgc2NhbGVGMTYgPSBVaW50MTZBcnJheS5mcm9tKHNjYWxlLCBmMTYpLCBpbnB1dEYxNiA9IFVpbnQxNkFycmF5LmZyb20oaW5wdXQsIGYxNikKICAgIGNvbnN0IHNjYWxlUSA9IEZsb2F0MzJBcnJheS5mcm9tKHNjYWxlRjE2LCBmcm9tRjE2KSwgaW5wdXRRID0gRmxvYXQzMkFycmF5LmZyb20oaW5wdXRGMTYsIGZyb21GMTYpCgogICAgLy8gQ1BVIHJlZmVyZW5jZSBmb3IgbWF0cml4IDA6IGJ5dGUgayBvZiB0ZXhlbCAoZHN0X3MsIHNyY19zKSBob2xkcyB0aGUgZm91cgogICAgLy8gMi1iaXQgd2VpZ2h0cyB0aGF0IGlucHV0IGNoYW5uZWwgayBtdWx0aXBsaWVzLCBvbmUgcGVyIG91dHB1dCBjaGFubmVsIGMuCiAgICBjb25zdCByZWYgPSBuZXcgRmxvYXQzMkFycmF5KE9VVF9TTElDRVMgKiA0KQogICAgY29uc3QgdzAgPSB3ZWlnaHRzWzBdCiAgICBmb3IgKGxldCBkID0gMDsgZCA8IE9VVF9TTElDRVM7IGQrKykgZm9yIChsZXQgYyA9IDA7IGMgPCA0OyBjKyspIHsKICAgICAgbGV0IGFjYyA9IDAKICAgICAgZm9yIChsZXQgcyA9IDA7IHMgPCBJTl9TTElDRVM7IHMrKykgZm9yIChsZXQgayA9IDA7IGsgPCA0OyBrKyspIHsgY29uc3QgYiA9IHcwWyhzICogT1VUX1NMSUNFUyArIGQpICogNCArIGtdOyBhY2MgKz0gaW5wdXRRW3MgKiA0ICsga10gKiAoKChiID4+ICgyICogYykpICYgMykgLSAyKSB9CiAgICAgIHJlZltkICogNCArIGNdID0gYWNjICogKEYzMiA/IHNjYWxlIDogc2NhbGVRKVtkICogNCArIGNdCiAgICB9CgogICAgLy8gR1BVIHJlc291cmNlcwogICAgY29uc3QgbWtUZXggPSAoYnl0ZXMpID0+IHsgY29uc3QgdCA9IGRldmljZS5jcmVhdGVUZXh0dXJlKHsgc2l6ZTogW09VVF9TTElDRVMsIElOX1NMSUNFU10sIGZvcm1hdDogJ3JnYmE4dWludCcsIHVzYWdlOiBHUFVUZXh0dXJlVXNhZ2UuVEVYVFVSRV9CSU5ESU5HIHwgR1BVVGV4dHVyZVVzYWdlLkNPUFlfRFNUIH0pOyBkZXZpY2UucXVldWUud3JpdGVUZXh0dXJlKHsgdGV4dHVyZTogdCB9LCBieXRlcywgeyBieXRlc1BlclJvdzogT1VUX1NMSUNFUyAqIDQgfSwgW09VVF9TTElDRVMsIElOX1NMSUNFU10pOyByZXR1cm4gdCB9CiAgICBjb25zdCBta0J1ZiA9IChieXRlcykgPT4geyBjb25zdCBiID0gZGV2aWNlLmNyZWF0ZUJ1ZmZlcih7IHNpemU6IGJ5dGVzLmJ5dGVMZW5ndGgsIHVzYWdlOiBHUFVCdWZmZXJVc2FnZS5TVE9SQUdFIHwgR1BVQnVmZmVyVXNhZ2UuQ09QWV9EU1QgfSk7IGRldmljZS5xdWV1ZS53cml0ZUJ1ZmZlcihiLCAwLCBieXRlcyk7IHJldHVybiBiIH0KICAgIGNvbnN0IHRleFcgPSB3ZWlnaHRzLm1hcCgodykgPT4gbWtUZXgodykpLCBidWZXID0gd2VpZ2h0cy5tYXAoKHcpID0+IG1rQnVmKHcpKQogICAgY29uc3Qgc2NhbGVCdWYgPSBta0J1ZihGMzIgPyBzY2FsZSA6IHNjYWxlRjE2KQogICAgY29uc3Qgc3JjVGV4ID0gZGV2aWNlLmNyZWF0ZVRleHR1cmUoeyBzaXplOiBbMSwgSU5fU0xJQ0VTXSwgZm9ybWF0OiAncmdiYTE2ZmxvYXQnLCB1c2FnZTogR1BVVGV4dHVyZVVzYWdlLlRFWFRVUkVfQklORElORyB8IEdQVVRleHR1cmVVc2FnZS5DT1BZX0RTVCB9KQogICAgZGV2aWNlLnF1ZXVlLndyaXRlVGV4dHVyZSh7IHRleHR1cmU6IHNyY1RleCB9LCBpbnB1dEYxNiwgeyBieXRlc1BlclJvdzogOCB9LCBbMSwgSU5fU0xJQ0VTXSkKICAgIGNvbnN0IGRzdFRleCA9IGRldmljZS5jcmVhdGVUZXh0dXJlKHsgc2l6ZTogWzEsIE9VVF9TTElDRVNdLCBmb3JtYXQ6ICdyZ2JhMTZmbG9hdCcsIHVzYWdlOiBHUFVUZXh0dXJlVXNhZ2UuU1RPUkFHRV9CSU5ESU5HIHwgR1BVVGV4dHVyZVVzYWdlLkNPUFlfU1JDIH0pCiAgICBjb25zdCB1bmkgPSBkZXZpY2UuY3JlYXRlQnVmZmVyKHsgc2l6ZTogMTYsIHVzYWdlOiBHUFVCdWZmZXJVc2FnZS5VTklGT1JNIHwgR1BVQnVmZmVyVXNhZ2UuQ09QWV9EU1QgfSkKICAgIGRldmljZS5xdWV1ZS53cml0ZUJ1ZmZlcih1bmksIDAsIG5ldyBJbnQzMkFycmF5KFtPVVRfU0xJQ0VTLCBJTl9TTElDRVMsIDAsIDBdKSkKICAgIGNvbnN0IHJlYWRCdWYgPSBkZXZpY2UuY3JlYXRlQnVmZmVyKHsgc2l6ZTogMjU2ICogT1VUX1NMSUNFUywgdXNhZ2U6IEdQVUJ1ZmZlclVzYWdlLkNPUFlfRFNUIHwgR1BVQnVmZmVyVXNhZ2UuTUFQX1JFQUQgfSkKICAgIGNvbnN0IE5RID0gMiAqIFJPVU5EUyAqIE5NQVRfQ09MRAogICAgY29uc3QgcXMgPSBkZXZpY2UuY3JlYXRlUXVlcnlTZXQoeyB0eXBlOiAndGltZXN0YW1wJywgY291bnQ6IE5RIH0pCiAgICBjb25zdCBxUmVzb2x2ZSA9IGRldmljZS5jcmVhdGVCdWZmZXIoeyBzaXplOiBOUSAqIDgsIHVzYWdlOiBHUFVCdWZmZXJVc2FnZS5RVUVSWV9SRVNPTFZFIHwgR1BVQnVmZmVyVXNhZ2UuQ09QWV9TUkMgfSkKICAgIGNvbnN0IHFSZWFkID0gZGV2aWNlLmNyZWF0ZUJ1ZmZlcih7IHNpemU6IE5RICogOCwgdXNhZ2U6IEdQVUJ1ZmZlclVzYWdlLkNPUFlfRFNUIHwgR1BVQnVmZmVyVXNhZ2UuTUFQX1JFQUQgfSkKCiAgICBjb25zdCByZWFkT3V0cHV0ID0gYXN5bmMgKCkgPT4gewogICAgICBjb25zdCBlbmMgPSBkZXZpY2UuY3JlYXRlQ29tbWFuZEVuY29kZXIoKQogICAgICBlbmMuY29weVRleHR1cmVUb0J1ZmZlcih7IHRleHR1cmU6IGRzdFRleCB9LCB7IGJ1ZmZlcjogcmVhZEJ1ZiwgYnl0ZXNQZXJSb3c6IDI1NiB9LCBbMSwgT1VUX1NMSUNFU10pCiAgICAgIGRldmljZS5xdWV1ZS5zdWJtaXQoW2VuYy5maW5pc2goKV0pCiAgICAgIGF3YWl0IHJlYWRCdWYubWFwQXN5bmMoR1BVTWFwTW9kZS5SRUFEKQogICAgICBjb25zdCB1MTYgPSBuZXcgVWludDE2QXJyYXkocmVhZEJ1Zi5nZXRNYXBwZWRSYW5nZSgpLnNsaWNlKDApKQogICAgICBjb25zdCBvdXQgPSBuZXcgRmxvYXQzMkFycmF5KE9VVF9TTElDRVMgKiA0KQogICAgICBmb3IgKGxldCBkID0gMDsgZCA8IE9VVF9TTElDRVM7IGQrKykgZm9yIChsZXQgYyA9IDA7IGMgPCA0OyBjKyspIG91dFtkICogNCArIGNdID0gZnJvbUYxNih1MTZbZCAqIDEyOCArIGNdKQogICAgICByZWFkQnVmLnVubWFwKCkKICAgICAgcmV0dXJuIG91dAogICAgfQoKICAgIC8vIFtuYW1lLCB3ZWlnaHRzIGZyb20gYnVmZmVyLCBLIHNwbGl0LCB3b3JrZ3JvdXAgeF0KICAgIGNvbnN0IFZBUklBTlRTID0gWwogICAgICBbJ3dhcm11cCcsIGZhbHNlLCA0LCAxNl0sCiAgICAgIFsnb3JpZycsIGZhbHNlLCA0LCAxNl0sCiAgICAgIFsnd2cyNTYtc2FtZScsIGZhbHNlLCA0LCA2NF0sCiAgICAgIFsnc3BsaXQxNi02NCcsIGZhbHNlLCAxNiwgNF0sCiAgICAgIFsnc3BsaXQzMi02NCcsIGZhbHNlLCAzMiwgMl0sCiAgICAgIFsnc3BsaXQxNi0yNTYnLCBmYWxzZSwgMTYsIDE2XSwKICAgICAgWydidWYtNCcsIHRydWUsIDQsIDE2XSwKICAgIF0KICAgIGNvbnN0IFYgPSB7fQogICAgZm9yIChjb25zdCBbbmFtZSwgYnVmLCBrcywgd2dYXSBvZiBWQVJJQU5UUykgewogICAgICBjb25zdCBkaXNwYXRjaFggPSBPVVRfU0xJQ0VTIC8gd2dYCiAgICAgIGNvbnN0IG1vZHVsZSA9IGRldmljZS5jcmVhdGVTaGFkZXJNb2R1bGUoeyBjb2RlOiB2YXJpYW50U291cmNlKGJhc2UsIGJ1Ziwga3MsIHdnWCkgfSkKICAgICAgY29uc3QgaW5mbyA9IGF3YWl0IG1vZHVsZS5nZXRDb21waWxhdGlvbkluZm8oKQogICAgICBjb25zdCBlcnJzID0gaW5mby5tZXNzYWdlcy5maWx0ZXIoKG0pID0+IG0udHlwZSA9PT0gJ2Vycm9yJykKICAgICAgaWYgKGVycnMubGVuZ3RoKSB0aHJvdyBuZXcgRXJyb3IoYCR7bmFtZX0gZmFpbGVkIHRvIGNvbXBpbGU6ICR7ZXJycy5tYXAoKG0pID0+IGAke20ubGluZU51bX06ICR7bS5tZXNzYWdlfWApLmpvaW4oJzsgJyl9YCkKICAgICAgY29uc3QgcGlwZWxpbmUgPSBkZXZpY2UuY3JlYXRlQ29tcHV0ZVBpcGVsaW5lKHsgbGF5b3V0OiAnYXV0bycsIGNvbXB1dGU6IHsgbW9kdWxlLCBlbnRyeVBvaW50OiAnbWFpbicsIGNvbnN0YW50czogeyAwOiB3Z1gsIDE6IGtzLCAyOiAxIH0gfSB9KQogICAgICBjb25zdCBiaW5kR3JvdXBzID0gd2VpZ2h0cy5tYXAoKF8sIG0pID0+IGRldmljZS5jcmVhdGVCaW5kR3JvdXAoewogICAgICAgIGxheW91dDogcGlwZWxpbmUuZ2V0QmluZEdyb3VwTGF5b3V0KDApLAogICAgICAgIGVudHJpZXM6IFsKICAgICAgICAgIHsgYmluZGluZzogMCwgcmVzb3VyY2U6IGRzdFRleC5jcmVhdGVWaWV3KCkgfSwKICAgICAgICAgIHsgYmluZGluZzogMSwgcmVzb3VyY2U6IHNyY1RleC5jcmVhdGVWaWV3KCkgfSwKICAgICAgICAgIHsgYmluZGluZzogMiwgcmVzb3VyY2U6IGJ1ZiA/IHsgYnVmZmVyOiBidWZXW21dIH0gOiB0ZXhXW21dLmNyZWF0ZVZpZXcoKSB9LAogICAgICAgICAgeyBiaW5kaW5nOiAzLCByZXNvdXJjZTogeyBidWZmZXI6IHNjYWxlQnVmIH0gfSwKICAgICAgICAgIHsgYmluZGluZzogNCwgcmVzb3VyY2U6IHsgYnVmZmVyOiB1bmkgfSB9LAogICAgICAgIF0sCiAgICAgIH0pKQogICAgICAvLyBjb3JyZWN0bmVzcyBvbiBtYXRyaXggMAogICAgICB7IGNvbnN0IGVuYyA9IGRldmljZS5jcmVhdGVDb21tYW5kRW5jb2RlcigpOyBjb25zdCBwID0gZW5jLmJlZ2luQ29tcHV0ZVBhc3MoKTsgcC5zZXRQaXBlbGluZShwaXBlbGluZSk7IHAuc2V0QmluZEdyb3VwKDAsIGJpbmRHcm91cHNbMF0pOyBwLmRpc3BhdGNoV29ya2dyb3VwcyhkaXNwYXRjaFgsIDEsIDEpOyBwLmVuZCgpOyBkZXZpY2UucXVldWUuc3VibWl0KFtlbmMuZmluaXNoKCldKSB9CiAgICAgIGNvbnN0IG91dCA9IGF3YWl0IHJlYWRPdXRwdXQoKQogICAgICBsZXQgbWF4QWJzID0gMCwgbWF4UmVmID0gMAogICAgICBmb3IgKGxldCBpID0gMDsgaSA8IG91dC5sZW5ndGg7IGkrKykgeyBtYXhBYnMgPSBNYXRoLm1heChtYXhBYnMsIE1hdGguYWJzKG91dFtpXSAtIHJlZltpXSkpOyBtYXhSZWYgPSBNYXRoLm1heChtYXhSZWYsIE1hdGguYWJzKHJlZltpXSkpIH0KICAgICAgVltuYW1lXSA9IHsgbmFtZSwgYnVmLCBrcywgd2dYLCBkaXNwYXRjaFgsIGludm9jYXRpb25zOiB3Z1ggKiBrcywgdGhyZWFkczogZGlzcGF0Y2hYICogd2dYICoga3MsIHBpcGVsaW5lLCBiaW5kR3JvdXBzLCBtYXhBYnNFcnI6IG1heEFicywgbWF4UmVmLCBvdXQsIGNvbGQ6IFtdLCBob3Q6IFtdIH0KICAgIH0KICAgIGZvciAoY29uc3QgdiBvZiBPYmplY3QudmFsdWVzKFYpKSB7IGxldCBkID0gMDsgZm9yIChsZXQgaSA9IDA7IGkgPCB2Lm91dC5sZW5ndGg7IGkrKykgZCA9IE1hdGgubWF4KGQsIE1hdGguYWJzKHYub3V0W2ldIC0gVi5vcmlnLm91dFtpXSkpOyB2LnZzT3JpZyA9IGQgfQogICAgZm9yIChjb25zdCB2IG9mIE9iamVjdC52YWx1ZXMoVikpIGRlbGV0ZSB2Lm91dAoKICAgIC8vIG9uZSB0aW1lZCBibG9jazogUk9VTkRTIHggbm1hdCBzaW5nbGUtZGlzcGF0Y2ggcGFzc2VzLCBlYWNoIHRpbWVzdGFtcGVkCiAgICBjb25zdCB0aW1lID0gYXN5bmMgKHYsIG5tYXQpID0+IHsKICAgICAgeyBjb25zdCBlbmMgPSBkZXZpY2UuY3JlYXRlQ29tbWFuZEVuY29kZXIoKTsgZm9yIChsZXQgbSA9IDA7IG0gPCBubWF0OyBtKyspIHsgY29uc3QgcCA9IGVuYy5iZWdpbkNvbXB1dGVQYXNzKCk7IHAuc2V0UGlwZWxpbmUodi5waXBlbGluZSk7IHAuc2V0QmluZEdyb3VwKDAsIHYuYmluZEdyb3Vwc1ttXSk7IHAuZGlzcGF0Y2hXb3JrZ3JvdXBzKHYuZGlzcGF0Y2hYLCAxLCAxKTsgcC5lbmQoKSB9IGRldmljZS5xdWV1ZS5zdWJtaXQoW2VuYy5maW5pc2goKV0pIH0KICAgICAgYXdhaXQgZGV2aWNlLnF1ZXVlLm9uU3VibWl0dGVkV29ya0RvbmUoKQogICAgICBjb25zdCBlbmMgPSBkZXZpY2UuY3JlYXRlQ29tbWFuZEVuY29kZXIoKQogICAgICBsZXQgcSA9IDAKICAgICAgZm9yIChsZXQgciA9IDA7IHIgPCBST1VORFM7IHIrKykgZm9yIChsZXQgbSA9IDA7IG0gPCBubWF0OyBtKyspIHsKICAgICAgICBjb25zdCBwID0gZW5jLmJlZ2luQ29tcHV0ZVBhc3MoeyB0aW1lc3RhbXBXcml0ZXM6IHsgcXVlcnlTZXQ6IHFzLCBiZWdpbm5pbmdPZlBhc3NXcml0ZUluZGV4OiBxLCBlbmRPZlBhc3NXcml0ZUluZGV4OiBxICsgMSB9IH0pCiAgICAgICAgcC5zZXRQaXBlbGluZSh2LnBpcGVsaW5lKTsgcC5zZXRCaW5kR3JvdXAoMCwgdi5iaW5kR3JvdXBzW21dKTsgcC5kaXNwYXRjaFdvcmtncm91cHModi5kaXNwYXRjaFgsIDEsIDEpOyBwLmVuZCgpOyBxICs9IDIKICAgICAgfQogICAgICBlbmMucmVzb2x2ZVF1ZXJ5U2V0KHFzLCAwLCBxLCBxUmVzb2x2ZSwgMCkKICAgICAgZW5jLmNvcHlCdWZmZXJUb0J1ZmZlcihxUmVzb2x2ZSwgMCwgcVJlYWQsIDAsIHEgKiA4KQogICAgICBkZXZpY2UucXVldWUuc3VibWl0KFtlbmMuZmluaXNoKCldKQogICAgICBhd2FpdCBxUmVhZC5tYXBBc3luYyhHUFVNYXBNb2RlLlJFQUQpCiAgICAgIGNvbnN0IHRzID0gbmV3IEJpZ1VpbnQ2NEFycmF5KHFSZWFkLmdldE1hcHBlZFJhbmdlKCkuc2xpY2UoMCkpCiAgICAgIHFSZWFkLnVubWFwKCkKICAgICAgY29uc3QgdXMgPSBbXQogICAgICBmb3IgKGxldCBpID0gMDsgaSA8IHE7IGkgKz0gMikgdXMucHVzaChOdW1iZXIodHNbaSArIDFdIC0gdHNbaV0pIC8gMWUzKQogICAgICByZXR1cm4gdXMKICAgIH0KCiAgICAvLyB3YXJtLXVwIHNsb3QgKGRpc2NhcmRlZCksIHRoZW4gZm9yd2FyZCwgdGhlbiByZXZlcnNlLiBUZW4gcm91bmRzOiBhIFQ0CiAgICAvLyB0b29rIHNldmVyYWwgaHVuZHJlZCBtcyBvZiBsb2FkIHRvIGxlYXZlIGl0cyBpZGxlIGNsb2Nrczsgb25lIHJvdW5kIHdhcwogICAgLy8gZW5vdWdoIG9uIHRoZSBNMiBNYXggYnV0IGlzIGNoZWFwIGVpdGhlciB3YXkuCiAgICBjb25zdCB0aW1lZCA9IFZBUklBTlRTLnNsaWNlKDEpLm1hcCgoeCkgPT4geFswXSkKICAgIGZvciAobGV0IGkgPSAwOyBpIDwgMTA7IGkrKykgeyBhd2FpdCB0aW1lKFYud2FybXVwLCBOTUFUX0NPTEQpOyBhd2FpdCB0aW1lKFYud2FybXVwLCAxKSB9CiAgICBmb3IgKGNvbnN0IG5hbWUgb2YgWy4uLnRpbWVkLCAuLi5bLi4udGltZWRdLnJldmVyc2UoKV0pIHsKICAgICAgY29uc3QgdiA9IFZbbmFtZV0KICAgICAgY29uc3QgYyA9IGF3YWl0IHRpbWUodiwgTk1BVF9DT0xEKSwgaCA9IGF3YWl0IHRpbWUodiwgMSkKICAgICAgdi5jb2xkLnB1c2goLi4uYyk7IHYuaG90LnB1c2goLi4uaCkKICAgICAgUi5vcmRlci5wdXNoKHsgbmFtZSwgY29sZE1lZGlhbjogbWVkaWFuKGMpLCBob3RNZWRpYW46IG1lZGlhbihoKSB9KQogICAgfQogICAgZm9yIChjb25zdCBuYW1lIG9mIHRpbWVkKSB7CiAgICAgIGNvbnN0IHYgPSBWW25hbWVdCiAgICAgIGNvbnN0IHN0YXQgPSAoYSkgPT4gKHsgbWVkaWFuOiBtZWRpYW4oYSksIHAxMDogcGN0KGEsIDAuMSksIHA5MDogcGN0KGEsIDAuOSksIG46IGEubGVuZ3RoIH0pCiAgICAgIFIudmFyaWFudHNbbmFtZV0gPSB7IGJ1Zjogdi5idWYsIGtzOiB2LmtzLCB3Z1g6IHYud2dYLCBkaXNwYXRjaFg6IHYuZGlzcGF0Y2hYLCBpbnZvY2F0aW9uczogdi5pbnZvY2F0aW9ucywgdGhyZWFkczogdi50aHJlYWRzLCBtYXhBYnNFcnI6IHYubWF4QWJzRXJyLCBtYXhSZWY6IHYubWF4UmVmLCB2c09yaWc6IHYudnNPcmlnLCBjb2xkOiBzdGF0KHYuY29sZCksIGhvdDogc3RhdCh2LmhvdCkgfQogICAgICBjb25zdCByID0gUi52YXJpYW50c1tuYW1lXQogICAgICBsb2coYCR7bmFtZS5wYWRFbmQoMTIpfSB3ZyAke1N0cmluZyh2LndnWCkucGFkU3RhcnQoMil9eCR7U3RyaW5nKHYua3MpLnBhZEVuZCgyKX0gPSR7U3RyaW5nKHYuaW52b2NhdGlvbnMpLnBhZFN0YXJ0KDQpfSBpbnYgIHRocmVhZHMgJHtTdHJpbmcodi50aHJlYWRzKS5wYWRTdGFydCg2KX0gIGNvbGQgJHtyLmNvbGQubWVkaWFuLnRvRml4ZWQoMSkucGFkU3RhcnQoNil9IMK1cyA9ICR7KFdFSUdIVF9CWVRFUyAvIChyLmNvbGQubWVkaWFuICogMWUtNikgLyAxZTkpLnRvRml4ZWQoMCkucGFkU3RhcnQoNCl9IEdCL3MgKHAxMCAke3IuY29sZC5wMTAudG9GaXhlZCgxKX0sIHA5MCAke3IuY29sZC5wOTAudG9GaXhlZCgxKX0pICBob3QgJHtyLmhvdC5tZWRpYW4udG9GaXhlZCgxKS5wYWRTdGFydCg2KX0gwrVzICBtYXh8ZXJyfCAke3YubWF4QWJzRXJyLnRvRXhwb25lbnRpYWwoMil9IHZzIG9yaWcgJHt2LnZzT3JpZy50b0V4cG9uZW50aWFsKDIpfWApCiAgICB9CiAgICBsb2coJ3Blci1wb3NpdGlvbiBjb2xkIG1lZGlhbnM6ICcgKyBSLm9yZGVyLm1hcCgobykgPT4gYCR7by5uYW1lfT0ke28uY29sZE1lZGlhbi50b0ZpeGVkKDEpfWApLmpvaW4oJyAnKSkKICB9IGNhdGNoIChlKSB7CiAgICBSLmVycm9yID0gU3RyaW5nKGU/LnN0YWNrID8/IGUpCiAgICBsb2coJ0VSUk9SOiAnICsgUi5lcnJvcikKICB9CiAgd2luZG93Ll9fbWJEb25lID0gdHJ1ZQo8L3NjcmlwdD4K","out/shaders/0112_unlabeled.wgsl":"ZW5hYmxlIGYxNjsKQGlkKDApIG92ZXJyaWRlIFdPUktHUk9VUF9TSVpFX1g6IGkzMjsKQGlkKDEpIG92ZXJyaWRlIFdPUktHUk9VUF9TSVpFX1k6IGkzMjsKQGlkKDIpIG92ZXJyaWRlIFdPUktHUk9VUF9TSVpFX1o6IGkzMjsKCmZuIFBhY2s0eDE2ZmxvYXQodiA6IHZlYzQ8ZjMyPikgLT4gdmVjMjx1MzI+IHsKICByZXR1cm4gdmVjMjx1MzI+KHBhY2syeDE2ZmxvYXQodi54eSksIHBhY2syeDE2ZmxvYXQodi56dykpOwp9CgpmbiBVbnBhY2s0eDE2ZmxvYXQodiA6IHZlYzI8dTMyPikgLT4gdmVjNDxmMzI+IHsKICByZXR1cm4gdmVjNDxmMzI+KHVucGFjazJ4MTZmbG9hdCh2LngpLCB1bnBhY2syeDE2ZmxvYXQodi55KSk7Cn0KQGdyb3VwKDApIEBiaW5kaW5nKDApIHZhciBkc3RfdGVuc29yX2ltYWdlMmQgOiB0ZXh0dXJlX3N0b3JhZ2VfMmQ8cmdiYTE2ZmxvYXQsIHdyaXRlPjsKQGdyb3VwKDApIEBiaW5kaW5nKDEpIHZhciBzcmNfdGVuc29yX2ltYWdlMmQgOiB0ZXh0dXJlXzJkPGYzMj47CkBncm91cCgwKSBAYmluZGluZygyKSB2YXIgd2VpZ2h0c19pbWFnZTJkIDogdGV4dHVyZV8yZDx1MzI+OwpzdHJ1Y3Qgd2VpZ2h0c19zY2FsZV9idWZmZXJfdmVjdG9yIHsKICBkYXRhOiBhcnJheTx2ZWM0PGYxNj4+LAp9OwpAZ3JvdXAoMCkgQGJpbmRpbmcoMykgdmFyPHN0b3JhZ2UsIHJlYWQ+IHdlaWdodHNfc2NhbGVfYnVmZmVyIDogd2VpZ2h0c19zY2FsZV9idWZmZXJfdmVjdG9yOwpzdHJ1Y3QgU2NhbGFycyB7CiAgaTAgOiB2ZWM0PGkzMj4sCn07CkBncm91cCgwKSBAYmluZGluZyg0KSB2YXI8dW5pZm9ybT4gVTogU2NhbGFyczsKdmFyPHdvcmtncm91cD4gdGVtcCA6IGFycmF5PHZlYzQ8ZjE2PiwgNjQ+OwpAY29tcHV0ZSBAd29ya2dyb3VwX3NpemUoV09SS0dST1VQX1NJWkVfWCwgV09SS0dST1VQX1NJWkVfWSwgV09SS0dST1VQX1NJWkVfWikKZm4gbWFpbihAYnVpbHRpbihnbG9iYWxfaW52b2NhdGlvbl9pZCkgcmVzZXJ2ZWRfZ2lkIDogdmVjMzx1MzI+LApAYnVpbHRpbihsb2NhbF9pbnZvY2F0aW9uX2lkKSByZXNlcnZlZF9saWQgOiB2ZWMzPHUzMj4sCkBidWlsdGluKHdvcmtncm91cF9pZCkgcmVzZXJ2ZWRfZ3JvdXBfaWQgOiB2ZWMzPHUzMj4pIHsKICB2YXIgZHN0X3MgOiBpMzI9IGkzMihyZXNlcnZlZF9naWQueCk7CiAgdmFyIGRzdF9lbmRfc2xpY2UgOiBpMzI9IFUuaTAueDsKICB2YXIgZHN0X3Nfd2dfb2Zmc2V0IDogaTMyPSBpMzIocmVzZXJ2ZWRfZ3JvdXBfaWQueCkgKiBXT1JLR1JPVVBfU0laRV9YOwogIGlmIChkc3Rfc193Z19vZmZzZXQgPj0gZHN0X2VuZF9zbGljZSkge3JldHVybjt9CiAgdmFyIHJfc3AwIDogdmVjNDxmMTY+PSB2ZWM0PGYxNj4oMC4wLCAwLjAsIDAuMCwgMC4wKTsKICB2YXIgdGlkIDogdmVjMjxpMzI+OwogIHRpZC54ID0gaTMyKHJlc2VydmVkX2xpZC54KTsKICB0aWQueSA9IGkzMihyZXNlcnZlZF9saWQueSk7CiAgaWYgKGRzdF9zIDwgVS5pMC54KSB7CiAgdmFyIHdfc2NhbGUgOiB2ZWM0PGYxNj49IHdlaWdodHNfc2NhbGVfYnVmZmVyLmRhdGFbKGRzdF9zKV07CiAgdmFyIHdfYmlhcyA6IHZlYzQ8ZjE2Pj0gLXdfc2NhbGUgKiAodmVjNDxmMTY+KDIuMCwgMi4wLCAyLjAsIDIuMCkpOwogIGZvciAodmFyIHNyY19zIDogaTMyPSB0aWQueTsgc3JjX3MgPCBVLmkwLnk7IHNyY19zICs9IDQpIHsKICAgIHZhciB2MCA6IHZlYzQ8ZjE2Pj0gdmVjNDxmMTY+KHRleHR1cmVMb2FkKHNyY190ZW5zb3JfaW1hZ2UyZCwgdmVjMjxpMzI+KCgwKSwgKCgwKSAqIFUuaTAueSArIChzcmNfcykpKSwgMCkpOwogICAgdmFyIHcwIDogdmVjNDxmMTY+O3ZhciB3MSA6IHZlYzQ8ZjE2Pjt2YXIgdzIgOiB2ZWM0PGYxNj47dmFyIHczIDogdmVjNDxmMTY+OwogICAgdmFyIHcgOiB2ZWM0PHUzMj49IHZlYzQ8dTMyPih0ZXh0dXJlTG9hZCh3ZWlnaHRzX2ltYWdlMmQsIHZlYzI8aTMyPigoZHN0X3MpLCAoc3JjX3MpKSwgMCkpOwogICAgCiAgewogIHZhciB3dDAgOiB2ZWM0PGYxNj49IHZlYzQ8ZjE2Pih3KSAqIGYxNigwLjI1KTsKICB2YXIgd3QxIDogdmVjNDxmMTY+PSBmbG9vcih3dDApOwogIHcwLnggPSAod3QwLnggLSB3dDEueCkgKiBmMTYoNC4wKTsKICB3MS54ID0gKHd0MC55IC0gd3QxLnkpICogZjE2KDQuMCk7CiAgdzIueCA9ICh3dDAueiAtIHd0MS56KSAqIGYxNig0LjApOwogIHczLnggPSAod3QwLncgLSB3dDEudykgKiBmMTYoNC4wKTsKICB3dDAgPSB3dDEgKiBmMTYoMC4yNSk7CiAgd3QxID0gZmxvb3Iod3QwKTsKICB3MC55ID0gKHd0MC54IC0gd3QxLngpICogZjE2KDQuMCk7CiAgdzEueSA9ICh3dDAueSAtIHd0MS55KSAqIGYxNig0LjApOwogIHcyLnkgPSAod3QwLnogLSB3dDEueikgKiBmMTYoNC4wKTsKICB3My55ID0gKHd0MC53IC0gd3QxLncpICogZjE2KDQuMCk7CiAgd3QwID0gd3QxICogZjE2KDAuMjUpOwogIHd0MSA9IGZsb29yKHd0MCk7CiAgdzAueiA9ICh3dDAueCAtIHd0MS54KSAqIGYxNig0LjApOwogIHcxLnogPSAod3QwLnkgLSB3dDEueSkgKiBmMTYoNC4wKTsKICB3Mi56ID0gKHd0MC56IC0gd3QxLnopICogZjE2KDQuMCk7CiAgdzMueiA9ICh3dDAudyAtIHd0MS53KSAqIGYxNig0LjApOwogIHcwLncgPSB3dDEueDsKICB3MS53ID0gd3QxLnk7CiAgdzIudyA9IHd0MS56OwogIHczLncgPSB3dDEudzsKICB9CjsKICAgIHcwID0gZm1hKHcwLCB3X3NjYWxlLCB3X2JpYXMpOwogICAgdzEgPSBmbWEodzEsIHdfc2NhbGUsIHdfYmlhcyk7CiAgICB3MiA9IGZtYSh3Miwgd19zY2FsZSwgd19iaWFzKTsKICAgIHczID0gZm1hKHczLCB3X3NjYWxlLCB3X2JpYXMpOwogICAgcl9zcDAgPSBmbWEodmVjNDxmMTY+KHYwLngsIHYwLngsIHYwLngsIHYwLngpLCB3MCwgcl9zcDApOwogICAgcl9zcDAgPSBmbWEodmVjNDxmMTY+KHYwLnksIHYwLnksIHYwLnksIHYwLnkpLCB3MSwgcl9zcDApOwogICAgcl9zcDAgPSBmbWEodmVjNDxmMTY+KHYwLnosIHYwLnosIHYwLnosIHYwLnopLCB3Miwgcl9zcDApOwogICAgcl9zcDAgPSBmbWEodmVjNDxmMTY+KHYwLncsIHYwLncsIHYwLncsIHYwLncpLCB3Mywgcl9zcDApOwogIH0gCiAgfSAKICB0ZW1wW3RpZC54ICogNCArIHRpZC55XSA9IHJfc3AwOwogIGZvciAodmFyIHlzdHJpZGUgOiBpMzI9IDQgLyAyOyB5c3RyaWRlID4gMDsgeXN0cmlkZSAvPSAyKSB7CiAgICB3b3JrZ3JvdXBCYXJyaWVyKCk7CiAgICBpZiAodGlkLnkgPCB5c3RyaWRlKSB7CiAgICAgIHJfc3AwICs9IHRlbXBbdGlkLnggKiA0ICsgdGlkLnkgKyB5c3RyaWRlXTsKICAgICAgdGVtcFt0aWQueCAqIDQgKyB0aWQueV0gPSByX3NwMDsKICAgIH0KICB9CiAgaWYgKGRzdF9zID49IFUuaTAueCkge3JldHVybjt9CiAgaWYgKHRpZC55ICE9IDApIHtyZXR1cm47fQogIHsKICB2YXIgcmVzX3ZhbHVlIDogdmVjNDxmMTY+PSB2ZWM0PGYxNj4ocl9zcDApOwogIHRleHR1cmVTdG9yZShkc3RfdGVuc29yX2ltYWdlMmQsIHZlYzI8aTMyPigoMCksICgoMCkgKiBVLmkwLnggKyAoZHN0X3MpKSksIHZlYzQ8ZjMyPihyZXNfdmFsdWUpKTsKICB9Cn0K","run.mjs":"Ly8gaGVhZGxlc3MgQ2hyb21lIG9uIGEgQ29sYWIgVDQsIHBlciBDaHJvbWUncyBjb2xhYi1oZWFkbGVzcyBndWlkZQppbXBvcnQgeyBjcmVhdGVTZXJ2ZXIgfSBmcm9tICdub2RlOmh0dHAnCmltcG9ydCB7IGNyZWF0ZVJlYWRTdHJlYW0sIHN0YXRTeW5jLCB3cml0ZUZpbGVTeW5jIH0gZnJvbSAnbm9kZTpmcycKaW1wb3J0IHsgam9pbiwgbm9ybWFsaXplLCBleHRuYW1lIH0gZnJvbSAnbm9kZTpwYXRoJwppbXBvcnQgcHVwcGV0ZWVyIGZyb20gJ3B1cHBldGVlcicKY29uc3QgUk9PVCA9IHByb2Nlc3MuY3dkKCkKY29uc3QgTUlNRSA9IHsgJy5odG1sJzogJ3RleHQvaHRtbDsgY2hhcnNldD11dGYtOCcsICcud2dzbCc6ICd0ZXh0L3BsYWluJyB9CmNvbnN0IHNlcnZlciA9IGNyZWF0ZVNlcnZlcigocmVxLCByZXMpID0+IHsKICBjb25zdCBmID0gam9pbihST09ULCBub3JtYWxpemUoZGVjb2RlVVJJQ29tcG9uZW50KG5ldyBVUkwocmVxLnVybCwgJ2h0dHA6Ly94JykucGF0aG5hbWUpKSkKICBsZXQgczsgdHJ5IHsgcyA9IHN0YXRTeW5jKGYpLnNpemUgfSBjYXRjaCB7IHJldHVybiByZXMud3JpdGVIZWFkKDQwNCkuZW5kKCkgfQogIHJlcy53cml0ZUhlYWQoMjAwLCB7ICdDb250ZW50LVR5cGUnOiBNSU1FW2V4dG5hbWUoZildID8/ICdhcHBsaWNhdGlvbi9vY3RldC1zdHJlYW0nLCAnQ29udGVudC1MZW5ndGgnOiBzIH0pCiAgY3JlYXRlUmVhZFN0cmVhbShmKS5waXBlKHJlcykKfSkKYXdhaXQgbmV3IFByb21pc2UoKHIpID0+IHNlcnZlci5saXN0ZW4oODkxNywgJzEyNy4wLjAuMScsIHIpKQpjb25zdCBicm93c2VyID0gYXdhaXQgcHVwcGV0ZWVyLmxhdW5jaCh7IGhlYWRsZXNzOiAnbmV3JywgYXJnczogWwogICctLW5vLXNhbmRib3gnLCAnLS11c2UtYW5nbGU9dnVsa2FuJywgJy0tZW5hYmxlLWZlYXR1cmVzPVZ1bGthbicsICctLWRpc2FibGUtdnVsa2FuLXN1cmZhY2UnLAogICctLWVuYWJsZS11bnNhZmUtd2ViZ3B1JywgJy0tZW5hYmxlLWRhd24tZmVhdHVyZXM9YWxsb3dfdW5zYWZlX2FwaXMsZGlzYWJsZV9hZGFwdGVyX2Jsb2NrbGlzdCcsCiAgJy0tZGlzYWJsZS1kYXduLWZlYXR1cmVzPXRpbWVzdGFtcF9xdWFudGl6YXRpb24nLCAnLS1pZ25vcmUtZ3B1LWJsb2NrbGlzdCcgXSwgcHJvdG9jb2xUaW1lb3V0OiA2MDAwMDAgfSkKY29uc3QgcGFnZSA9IGF3YWl0IGJyb3dzZXIubmV3UGFnZSgpCnBhZ2Uub24oJ2NvbnNvbGUnLCAobSkgPT4geyBjb25zdCB0ID0gbS50ZXh0KCk7IGlmICh0LnN0YXJ0c1dpdGgoJ1ttYl0nKSkgY29uc29sZS5sb2codCkgfSkKLy8gQ2hyb21lIG9uIExpbnV4L05WSURJQSAoVnVsa2FuKSBtYXkgd2l0aGhvbGQgc2hhZGVyLWYxNiBldmVuIHdoZW4gdGhlIGRyaXZlciBoYXMgaXQ7Ci8vIHRoZSBwYWdlIHRoZW4gcnVucyBhbiBmMzIgdHJhbnNjcmlwdGlvbiBvZiB0aGUgc2FtZSBrZXJuZWwgKD9mMzI9MSkgYW5kIHNheXMgc28uCmxldCBSCmZvciAoY29uc3QgcSBvZiBbJycsICc/ZjMyPTEnXSkgewogIGF3YWl0IHBhZ2UuZ290bygnaHR0cDovLzEyNy4wLjAuMTo4OTE3L21pY3JvYmVuY2guaHRtbCcgKyBxLCB7IHdhaXRVbnRpbDogJ2xvYWQnIH0pCiAgYXdhaXQgcGFnZS53YWl0Rm9yRnVuY3Rpb24oJ3dpbmRvdy5fX21iRG9uZSA9PT0gdHJ1ZScsIHsgdGltZW91dDogNjAwMDAwLCBwb2xsaW5nOiA1MDAgfSkKICBSID0gYXdhaXQgcGFnZS5ldmFsdWF0ZSgoKSA9PiB3aW5kb3cuX19tYikKICBpZiAoIShSLmVycm9yICYmIC9sYWNrcyBzaGFkZXItZjE2Ly50ZXN0KFIuZXJyb3IpKSkgYnJlYWsKICBjb25zb2xlLmxvZygnbm8gc2hhZGVyLWYxNiBvbiB0aGlzIGFkYXB0ZXI7IHJlcnVubmluZyBhcyBmMzInKQp9ClIuZGF0ZSA9IG5ldyBEYXRlKCkudG9JU09TdHJpbmcoKQpSLmdwdUluZm8gPSBhd2FpdCBwYWdlLmV2YWx1YXRlKGFzeW5jICgpID0+IHsgY29uc3QgYSA9IGF3YWl0IG5hdmlnYXRvci5ncHUucmVxdWVzdEFkYXB0ZXIoKTsgcmV0dXJuIHsgdmVuZG9yOiBhPy5pbmZvPy52ZW5kb3IsIGFyY2hpdGVjdHVyZTogYT8uaW5mbz8uYXJjaGl0ZWN0dXJlLCBkZXZpY2U6IGE/LmluZm8/LmRldmljZSwgZGVzY3JpcHRpb246IGE/LmluZm8/LmRlc2NyaXB0aW9uLCBmZWF0dXJlczogWy4uLihhPy5mZWF0dXJlcyA/PyBbXSldIH0gfSkKd3JpdGVGaWxlU3luYygnbWljcm9iZW5jaC1jb2xhYi5qc29uJywgSlNPTi5zdHJpbmdpZnkoUiwgbnVsbCwgMikpCmNvbnNvbGUubG9nKCdBREFQVEVSJywgSlNPTi5zdHJpbmdpZnkoUi5ncHVJbmZvKSkKaWYgKFIuZXJyb3IpIGNvbnNvbGUubG9nKCdQQUdFIEVSUk9SJywgUi5lcnJvcikKY29uc29sZS5sb2coJ3NhdmVkIG1pY3JvYmVuY2gtY29sYWIuanNvbjsgY29weSBpdCBiYWNrIGludG8gb3V0LyBvZiB0aGUgbGl0ZXJ0LWNhcHR1cmUgZm9sZGVyJykKYXdhaXQgYnJvd3Nlci5jbG9zZSgpOyBzZXJ2ZXIuY2xvc2UoKQo="}
os.makedirs('kernel0112/out/shaders', exist_ok=True)
for name, data in files.items():
    with open(os.path.join('kernel0112', name), 'wb') as f:
        f.write(base64.b64decode(data))
!cd kernel0112 && npm init -y > /dev/null && npm i --silent puppeteer@25.11.0 > /dev/null && echo installed

In [ ]:
# 3) Run. Prints one line per variant; the JSON is the record to copy back.
!cd kernel0112 && node run.mjs
print(open('kernel0112/microbench-colab.json').read()[:600])

## Reading the result

`orig` is Google's kernel as shipped (16 × 4 workgroups, 12,288 threads). `wg256-same` changes only the workgroup size (control). `split16-64` and `split32-64` keep 64-thread workgroups and dispatch 4x and 8x the threads. On the M2 Max: orig 60.9 µs, control 60.6, split32 36.8 (1.65x). Whether the T4 shows the same shape is the question this notebook answers. Download `kernel0112/microbench-colab.json` and drop it into `out/`.